# XRF55 Bench — WavDualMambaV2

**Model:** WavDualMambaV2 — upgraded architecture with single-trunk backbone, temporal downsampling, context-aware pooling, and optional final attention.  
**Dataset:** XRF55 — 11 activities, 4620 train / 1980 test  
**Split:** train = reps 1–14, test = reps 15–20. No val.

Key changes vs WavDualMamba v1:
- **C1 `arch='trunk'`** (default): concat 3 subbands after stems → 1 shared backbone (vs 3 parallel branches). Cross-subband mixing at every TFBlock, not just at AdaptiveFusion.
- **C2 `downsample=True`**: learnable Conv2d stride-2 after TFBlocks → T 500→250 (~2× faster Mamba).
- **C3 `d_state=16`**: Mamba-1 default, saves params vs v1's 32.
- **C4 `ffn_ratio=2`**: FFN sub-block after each Mamba layer, ON by default.
- **C5 `d_model=96`**: wider features using budget freed by C1+C3.
- **C6 `use_final_attn=True`**: 1 MHSA layer after Mamba (at T=250, less data-hungry).
- **C7 `use_eca=True`**: ECA channel gate (~5 params) on concatenated stem outputs.
- **C8**: AttnStatPool upgraded to full ECAPA context-aware form.

---

## Attached dataset

| Dataset name | Mount path |
|---|---|
| `xrf55-amp-dataset` | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

## Protocol reference

| # | Optimizer | LR | WD | Scheduler | Epochs | Notes |
|---|---|---|---|---|---|---|
| 01 | AdamW | 1e-4 | 0.01 | None | 40 | tf_mamba paper |
| 02 | Adam | 1e-3 | 0.0 | MultiStepLR [40,80,120,160] γ=0.5 | 200 | XRF55 paper |
| 03 | AdamW | 5e-4 | 1e-3 | warmup(10ep)+cosine → 4e-5 | 200 | APWMamba paper |

## Ablation flags (`model_kwargs`)

### Architecture mode
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `arch` | `'trunk'` | `'trunk'` / `'branch'` | Single trunk (C1) or 3 parallel branches (legacy v1) |
| `subbands` | `('LL','HL','LH')` | any subset | Which DWT subbands to use |

### CNN stage
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `temporal_stride` | `1` | `1` / `2` | Stride in stem conv (2 → T 500→250 at stem) |
| `dilations` | `(1, 2)` | tuple | TFBlock dilation schedule |
| `dp_cnn` | `(0.0, 0.05)` | tuple | DropPath per TFBlock |
| `downsample` | `True` | `True` / `False` | Learnable T/2 downsample after TFBlocks (C2) |
| `use_post` | `True` | `True` / `False` | Post-downsample TFBlock refine step (trunk only) |
| `use_eca` | `True` | `True` / `False` | ECA channel gate after subband concat (C7) |

### Embed
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `d_mid` | `64` | int | Trunk channels after downsample |
| `embed_drop` | `0.1` | float | Dropout after Linear embed |
| `embed_hidden` | `None` | `int` / `None` | Two-stage embed hidden dim |

### Mamba
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `d_model` | `96` | int | Feature width entering Mamba (C5) |
| `d_state` | `16` | int | SSM state size (C3); 32 = v1 |
| `ffn_ratio` | `2` | `0` / `2` | FFN sub-block after each Mamba layer (C4) |
| `bidirectional` | `True` | `True` / `False` | Gated backward Mamba branch |
| `dp_mamba` | `(0.05, 0.10)` | tuple | DropPath per BiMamba layer |
| `expand` | `2` | int | Mamba inner-expansion |
| `d_conv` | `4` | int | Mamba local conv width |

### Fusion / Attention
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `use_final_attn` | `True` | `True` / `False` | MHSA layer after Mamba, before pool (C6) |
| `attn_heads` | `4` | int | Number of attention heads |
| `share_branches` | `False` | `True` / `False` | Tied branch weights (branch mode only) |
| `freq_mix` | `None` | `None` / `'mlp'` | Nonlinear subcarrier mix |
| `use_pos_emb` | `False` | `True` / `False` | Sinusoidal PE |

## Param budget (default trunk config, ~0.57M)

| Component | Params | Note |
|---|---|---|
| 3 × SubbandStem | ~11K | Per-subband physical kernels |
| TFBlock×2 (48-ch) | ~6K | On concatenated stems |
| TemporalDownsample | ~12.5K | 48→64, T/2 |
| TFBlock_post (64-ch) | ~5K | At reduced resolution (`use_post=True`) |
| ECA | ~5 | Negligible |
| Embed Linear(960→96) | ~92K | 64ch×15freq=960 |
| BiMamba×2 (d=96, FFN) | ~384K | 67% of total |
| FinalAttention | ~37K | 4-head MHSA |
| AttnStatPool | ~18.6K | Context-aware ECAPA |
| Classifier | ~2.5K | LN+Drop+Linear |
| **Total** | **~570K** | vs v1 ~582K |

In [ ]:
# Cell 1 — Install mamba-ssm (required by WavDualMambaV2 BiMamba layers)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Clone / update latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/xrf55_benchmark')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/xrf55_benchmark.git',
         str(CODE_PATH)],
        check=True
    )
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Configuration
from pathlib import Path
from xrf55_bench.config import TrainCfg, TrainCfg_for_protocol

SEEDS      = [42]               # single-seed default
# SEEDS    = [0, 4, 8, 17, 42] # multi-seed

# Data source — uncomment one:
BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/raw')
# BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/processed')

# ── Model kwargs — chỉnh trực tiếp ──────────────────────────────────────────
MODEL_KWARGS = {
    # ── Architecture mode ───────────────────────────────────────────────────
    'arch'            : 'trunk',             # 'branch' → legacy v1 layout (ablation)
    'subbands'        : ('LL', 'HL', 'LH'),  # ('HL','LH') | ('LL','HL') | ('LL','LH')

    # ── CNN stage ───────────────────────────────────────────────────────────
    'temporal_stride' : 1,                   # 2 → additional T/2 at stem
    'dilations'       : (1, 2),              # (1,2,4) → 3 TFBlocks
    'dp_cnn'          : (0.0, 0.05),
    'downsample'      : True,                # False → skip TemporalDownsample (C2)
    'use_post'        : True,                # False → skip post-downsample TFBlock
    'use_eca'         : True,                # False → ablate ECA gate (C7)

    # ── Embed ───────────────────────────────────────────────────────────────
    'd_mid'           : 64,                  # trunk channels after downsample
    'embed_drop'      : 0.1,
    'embed_hidden'    : None,                # int → two-stage embed

    # ── Mamba ───────────────────────────────────────────────────────────────
    'd_model'         : 96,                  # 64 → match v1 budget (ablation)
    'd_state'         : 16,                  # 32 → v1 state size (ablation C3)
    'ffn_ratio'       : 2,                   # 0 → no FFN (ablation C4)
    'bidirectional'   : True,
    'dp_mamba'        : (0.05, 0.10),
    'expand'          : 2,
    'd_conv'          : 4,

    # ── Attention / Fusion ──────────────────────────────────────────────────
    'use_final_attn'  : True,                # False → ablate FinalAttention (C6)
    'attn_heads'      : 4,
    'share_branches'  : False,               # branch mode only
    'freq_mix'        : None,                # 'mlp' → nonlinear subcarrier mix
    'use_pos_emb'     : False,
}

# ── Auto output-dir tag từ ablation config ───────────────────────────────────
_sb   = '_'.join(MODEL_KWARGS.get('subbands', ('LL', 'HL', 'LH'))).lower()
_tag  = _sb
if MODEL_KWARGS.get('arch', 'trunk') == 'branch':          _tag += '_branch'
if not MODEL_KWARGS.get('downsample', True):                _tag += '_nodown'
if not MODEL_KWARGS.get('use_post', True):                  _tag += '_nopost'
if not MODEL_KWARGS.get('use_final_attn', True):            _tag += '_noattn'
if not MODEL_KWARGS.get('use_eca', True):                   _tag += '_noeca'
if MODEL_KWARGS.get('d_model', 96) != 96:                   _tag += f"_d{MODEL_KWARGS['d_model']}"
if MODEL_KWARGS.get('d_state', 16) != 16:                   _tag += f"_s{MODEL_KWARGS['d_state']}"
if MODEL_KWARGS.get('ffn_ratio', 2) != 2:                   _tag += f"_ffn{MODEL_KWARGS['ffn_ratio']}"
if MODEL_KWARGS.get('temporal_stride', 1) == 2:             _tag += '_t250'
if MODEL_KWARGS.get('share_branches'):                      _tag += '_shared'
if MODEL_KWARGS.get('freq_mix'):                            _tag += f"_{MODEL_KWARGS['freq_mix']}"
if MODEL_KWARGS.get('use_pos_emb'):                         _tag += '_pe'
if MODEL_KWARGS.get('bidirectional') is False:              _tag += '_uni'
if MODEL_KWARGS.get('embed_hidden'):                        _tag += f"_emb{MODEL_KWARGS['embed_hidden']}"
OUTPUT_DIR = Path(f'/kaggle/working/outputs/wavdual_v2_{_tag}_p02_raw')

print(f'Seeds      : {SEEDS}')
print(f'Bench dir  : {BENCH_DIR}')
print(f'Model cfg  : {MODEL_KWARGS}')
print(f'Output dir : {OUTPUT_DIR}')

for fname in ['stats.json', 'y_train.npy', 'y_test.npy',
              'wavmamba/X_train.npy', 'wavmamba/X_test.npy']:
    p = BENCH_DIR / fname
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')

In [ ]:
# Cell 4 — Run training
#
# Protocol presets:
#   01  AdamW  lr=1e-4  wd=0.01  no scheduler    40 ep   (tf_mamba paper)
#   02  Adam   lr=1e-3  wd=0.0   MultiStepLR     200 ep  (XRF55 paper)
#             milestones=[40,80,120,160]  gamma=0.5
#   03  AdamW  lr=5e-4  wd=1e-3  warmup+cosine   200 ep  (APWMamba paper)
#             warmup=10ep  floor_lr=4e-5  label_smoothing=0.0
#
# LR overrides (None = keep the protocol default):
MAX_LR   = None     # e.g. 3e-4
FLOOR_LR = None     # e.g. 1e-5
_lr_over = {k: v for k, v in (('lr', MAX_LR), ('floor_lr', FLOOR_LR)) if v is not None}

# Uncomment one cfg line:
# cfg = TrainCfg_for_protocol('01', seeds=tuple(SEEDS), **_lr_over)          # 200ep, Adam  lr=1e-3, MultiStepLR
# cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS), **_lr_over)          # 200ep, AdamW lr=5e-4, warmup+cosine
cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS), **_lr_over)            # 200ep, AdamW lr=5e-4, warmup+cosine

run(model_name='wavdualmamba_v2', bench_dir=BENCH_DIR, output_dir=OUTPUT_DIR,
    train_cfg=cfg, num_workers=4, model_kwargs=MODEL_KWARGS)

In [ ]:
# Cell 4b (optional) — Ablation sweep: trunk vs branch, with/without FinalAttention
#
# ABLATIONS = [
#     {'arch': 'trunk',  'use_final_attn': True,  '_name': 'trunk_attn'},
#     {'arch': 'trunk',  'use_final_attn': False, '_name': 'trunk_noattn'},
#     {'arch': 'branch', 'use_final_attn': False, 'd_model': 64, 'd_state': 32,
#      'ffn_ratio': 0, 'dp_mamba': (0.0, 0.10), '_name': 'branch_v1'},
# ]
# for ab in ABLATIONS:
#     name = ab.pop('_name')
#     kw   = {**MODEL_KWARGS, **ab}
#     out  = Path(f'/kaggle/working/outputs/wavdual_v2_{name}_p02_raw')
#     cfg  = TrainCfg_for_protocol('02', seeds=tuple(SEEDS))
#     print(f'\n########## {name} -> {out} ##########')
#     run(model_name='wavdualmamba_v2', bench_dir=BENCH_DIR, output_dir=out,
#         train_cfg=cfg, num_workers=4, model_kwargs=kw)

In [ ]:
# Cell 5 — Results
import json

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    s     = m['summary']
    seeds = m['config']['seeds']
    print(f"\n{'='*55}")
    print(f"  XRF55 Bench — WavDualMambaV2")
    print(f"  Model cfg   : {m.get('model_config', {})}")
    print(f"  Seeds       : {seeds}")
    print(f"  Accuracy    : {s['test_accuracy_mean']*100:.2f}% ± {s['test_accuracy_std']*100:.2f}%")
    print(f"  F1 Macro    : {s['test_f1_macro_mean']*100:.2f}% ± {s['test_f1_macro_std']*100:.2f}%")
    print(f"  Best epochs : {s['best_epochs']}")
    print(f"  Params      : {s['params_M']}M  |  Size: {s['model_size_mb']}MB")
    if s.get('macs_G'):
        print(f"  MACs        : {s['macs_G']:.3f}G")
    if s.get('latency_mean_ms') is not None:
        print(f"  Latency     : {s['latency_mean_ms']:.2f} ± {s['latency_std_ms']:.2f} ms")
    print(f"  Time        : {s['total_time_s']}s")
    print(f"{'='*55}")

    # run_config.json summary
    rc_path = OUTPUT_DIR / 'run_config.json'
    if rc_path.exists():
        with open(rc_path) as f:
            rc = json.load(f)
        r = rc.get('results', {})
        print(f"\n  Headline    : {r.get('headline')}")
        print(f"  Final epoch : {r.get('final_epoch')}")
        print(f"  Best epoch  : {r.get('best_epoch')} [DIAGNOSTIC only]")

    if len(seeds) == 1:
        ps = m['per_seed'].get(str(seeds[0]), {})
        if ps.get('test_f1_per_class'):
            class_names = [
                'Waving', 'Clap Hands', 'Fall on the Floor', 'Jumping', 'Running',
                'Sitting Down', 'Standing Up', 'Turning', 'Walking',
                'Stretch Oneself', 'Pat on Shoulder',
            ]
            print(f"\n  Per-class F1:")
            for i, v in enumerate(ps['test_f1_per_class']):
                print(f"    {class_names[i]:<20}: {v*100:.2f}%")
else:
    print('metrics.json not found — training may not have completed.')

In [ ]:
# Cell 6 — Plots + Download
import shutil
from IPython.display import Image, display, FileLink

for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
    p = OUTPUT_DIR / 'plots' / fname
    if p.exists():
        display(Image(str(p)))

print()
print('--- Download ---')
zips = sorted(OUTPUT_DIR.glob('*.zip'))
if zips:
    for src in zips:
        dst = Path('/kaggle/working') / src.name
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / 1e6
        print(f'{src.name}  ({size_mb:.1f} MB)')
        display(FileLink(src.name))
else:
    print('[MISSING] no zip found — run Cell 4 first.')